# 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Destination folder in your Drive
DESTINATION = "/content/drive/MyDrive/ColabMedia"

# Create folder if it doesn't exist
import os
os.makedirs(DESTINATION, exist_ok=True)

# 2) Install modern multimedia extraction tool

In [ ]:
!sudo apt-get remove -y yt-dlp

In [ ]:
# Install latest yt-dlp with EJS support
# yt-dlp[default] pulls in yt-dlp-ejs at the exact version pinned by this
# yt-dlp build's pyproject.toml — a standalone `pip install yt-dlp-ejs` grabs
# latest-on-PyPI instead, which can silently mismatch a master.zip build.
!pip install -U "yt-dlp[default] @ https://github.com/yt-dlp/yt-dlp/archive/master.zip"

# Install ipywidgets
!pip install -q ipywidgets

from IPython.display import display
import ipywidgets as widgets
import os

# yt-dlp requires Node >= 22.0.0 to run the EJS/PO-token solver. Ubuntu's
# apt 'nodejs' package ships 18.x — present and on PATH, but yt-dlp silently
# treats it as unsupported ("No supported JavaScript runtime could be found")
# even with --js-runtimes node passed. Use NodeSource's setup script instead.
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y nodejs


In [ ]:
# Verify installation — node must report >= 22, otherwise yt-dlp will
# silently ignore it and fall back to the deno-only default.
!node -v
!yt-dlp --version
!pip show yt-dlp-ejs 2>&1 | grep -i version

In [ ]:
# symbolic link (-f: idempotent on reruns — a stale link here silently
# breaks --js-runtimes node without raising, which is worse than the
# "File exists" warning it replaces)
!ln -sf /usr/bin/node /usr/local/bin/node

# 3) Create UI for cookie upload

In [ ]:
# UI: Upload cookies.txt
upload_label = widgets.Label("Upload your cookies.txt file:")
upload_button = widgets.FileUpload(
    accept='.txt',
    multiple=False,
    description='Upload cookies.txt'
)

display(upload_label)
display(upload_button)

# Global path for cookies.txt
cookies_txt_path = "/content/cookies.txt"

def save_uploaded_file(change):
    global cookies_txt_path

    if upload_button.value:
        # Extract file info
        file_info = list(upload_button.value.values())[0]
        file_content = file_info['content']

        # Save file to Colab
        with open(cookies_txt_path, "wb") as f:
            f.write(file_content)

        print(f"✔ cookies.txt saved at: {cookies_txt_path}")

upload_button.observe(save_uploaded_file, names='value')


# 4) UI for entering the video URL

In [ ]:
url_label = widgets.Label("Enter the video URL:")
url_box = widgets.Text(
    value='https://www.youtube.com/watch?v=xxxxxx',
    description='URL:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

display(url_label)
display(url_box)

# 3) Process a link and save the file to Drive

In [ ]:
def process_video(b):
    # Check if the cookies file actually exists
    if not os.path.exists(cookies_txt_path):
        print(f"❌ No cookies file found at {cookies_txt_path}. Please upload it.")
        return

    url = url_box.value.strip()
    if not url:
        print("❌ Please enter a valid URL.")
        return

    print("Processing with EJS challenge solver...")

    # --js-runtimes node picks the runtime; --remote-components ejs:github is
    # a fallback so extraction still works if the pip-installed yt-dlp-ejs
    # version lags behind this yt-dlp build (it only fetches remotely when
    # yt-dlp actually needs to, per the flag's own docs).
    cmd = (
        f'yt-dlp --js-runtimes node --remote-components ejs:github '
        f'--cookies "{cookies_txt_path}" -o "{DESTINATION}/%(title)s.%(ext)s" "{url}"'
    )
    !$cmd

    print("✔ File processed and saved to your Google Drive:", DESTINATION)


process_button = widgets.Button(
    description="Start Processing",
    button_style='success'
)

process_button.on_click(process_video)
display(process_button)

# Automatically trigger the process_video function after setting the URL
process_video(None)